# Stack Assembly — Build the 12-Channel Multi-Source Stack (4-Tile Rebuild)

**REVISED FOR MEMORY SAFETY.** The mosaic is ~22,000 x 21,000 pixels.
A single float32 band at that size is ~1.9GB; holding 6+ bands plus
indices plus the final stack in RAM simultaneously exceeds Colab's
available memory and crashes the runtime (confirmed in testing).

This version writes each resampled band to disk immediately rather than
keeping it in memory, then builds the final stack using windowed
(chunked) processing — reading and writing small tiles of the array at a
time instead of the whole thing at once.

**Project:** Deep Learning for Flood Inundation Mapping Using Multi-Source Satellite Data
**Study Area:** KwaZulu-Natal, South Africa (36JTM, 36JUM, 36JTN, 36JUN)
**Flood Event:** April 2022 KwaZulu-Natal Floods
**Author:** Valencia
**Supervisor:** Prof. Innocent Davidson
**Institution:** Cape Peninsula University of Technology (CPUT)

---

## Channel order (matches original pipeline)

| Channel | Band | Description |
|---------|------|-------------|
| 0 | S2 B02 | Blue |
| 1 | S2 B03 | Green |
| 2 | S2 B04 | Red |
| 3 | S2 B08 | Near-Infrared |
| 4 | S2 B11 | SWIR-1 |
| 5 | S2 B12 | SWIR-2 |
| 6 | NDVI | Vegetation index |
| 7 | NDWI | Water index |
| 8 | MNDWI | Modified water index |
| 9 | S1 VV-pre | Pre-flood VV backscatter |
| 10 | S1 VV-post | Post-flood VV backscatter |
| 11 | DEM | Elevation (reprojected, normalised) |

---
## Step 1: Mount Drive and Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install rasterio --quiet

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window
import warnings
warnings.filterwarnings('ignore')

ROOT = '/content/drive/MyDrive/KZN_Research_Colab/'
MOSAIC_DIR = ROOT + 'Mosaic_4tile/'
S2_DIR = MOSAIC_DIR + 'S2_bands/'
OUT_DIR = ROOT + 'Stacked_4tile/'
ALIGNED_DIR = OUT_DIR + 'Aligned_bands/'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(ALIGNED_DIR, exist_ok=True)

REFERENCE_PATH = MOSAIC_DIR + 'S1_mosaic_post.tif'

with rasterio.open(REFERENCE_PATH) as ref:
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_width = ref.width
    ref_height = ref.height

print('Reference grid (from S1 post-flood mosaic):')
print(f'  CRS   : {ref_crs}')
print(f'  Shape : {ref_height} x {ref_width}')
print(f'  Estimated size per band (float32): {ref_height*ref_width*4/1e9:.2f} GB')

Reference grid (from S1 post-flood mosaic):
  CRS   : EPSG:32736
  Shape : 22139 x 21576
  Estimated size per band (float32): 1.91 GB


---
## Step 2: Resample Each Source to Disk (One Band at a Time)

Each input is reprojected/resampled onto the reference grid and written
**directly to a GeoTIFF on disk** — never held as a full array in Python
memory for longer than the single `reproject()` call needs. This keeps
peak memory use to roughly one band's worth at a time, not all of them.

In [ ]:
def resample_to_reference_file(src_path, dst_path, resampling=Resampling.bilinear, band_index=1):
    """
    Resample one band from src_path onto the reference grid and write
    the result straight to dst_path. Does not return the array — caller
    should re-open dst_path if the data is needed.
    """
    with rasterio.open(src_path) as src:
        meta = {
            'driver': 'GTiff', 'dtype': 'float32', 'count': 1,
            'height': ref_height, 'width': ref_width,
            'crs': ref_crs, 'transform': ref_transform, 'nodata': None
        }
        with rasterio.open(dst_path, 'w', **meta) as dst:
            reproject(
                source=rasterio.band(src, band_index),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                resampling=resampling
            )
    return dst_path

print('Helper function defined.')

Helper function defined.


In [ ]:
S2_BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']

print('Resampling Sentinel-2 bands to reference grid (writing to disk one at a time)...\n')
for band in S2_BANDS:
    src_path = S2_DIR + f'S2_{band}_post.tif'
    dst_path = ALIGNED_DIR + f'S2_{band}_aligned.tif'
    if os.path.exists(dst_path):
        print(f'  [SKIP - already done] {band}')
        continue
    if not os.path.exists(src_path):
        print(f'  [MISSING] {band}: {src_path}')
        continue
    resample_to_reference_file(src_path, dst_path, resampling=Resampling.bilinear)
    print(f'  [OK] {band} -> {dst_path}')

print('\nDone ')

Resampling Sentinel-2 bands to reference grid (writing to disk one at a time)...

  [OK] B02 -> /content/drive/MyDrive/KZN_Research_Colab/Stacked_4tile/Aligned_bands/S2_B02_aligned.tif
  [OK] B03 -> /content/drive/MyDrive/KZN_Research_Colab/Stacked_4tile/Aligned_bands/S2_B03_aligned.tif
  [OK] B04 -> /content/drive/MyDrive/KZN_Research_Colab/Stacked_4tile/Aligned_bands/S2_B04_aligned.tif
  [OK] B08 -> /content/drive/MyDrive/KZN_Research_Colab/Stacked_4tile/Aligned_bands/S2_B08_aligned.tif
  [OK] B11 -> /content/drive/MyDrive/KZN_Research_Colab/Stacked_4tile/Aligned_bands/S2_B11_aligned.tif
  [OK] B12 -> /content/drive/MyDrive/KZN_Research_Colab/Stacked_4tile/Aligned_bands/S2_B12_aligned.tif

Done. Each band written separately 


In [ ]:
print('Resampling Sentinel-1 (pre and post VV) ...\n')

s1_targets = [
    (MOSAIC_DIR + 'S1_mosaic_pre.tif',  ALIGNED_DIR + 'S1_VV_pre_aligned.tif'),
    (MOSAIC_DIR + 'S1_mosaic_post.tif', ALIGNED_DIR + 'S1_VV_post_aligned.tif'),
]

for src_path, dst_path in s1_targets:
    if os.path.exists(dst_path):
        print(f'  [SKIP - already done] {os.path.basename(dst_path)}')
        continue
    resample_to_reference_file(src_path, dst_path, resampling=Resampling.bilinear, band_index=1)
    print(f'  [OK] -> {dst_path}')

Resampling Sentinel-1 (pre and post VV) ...

  [OK] -> /content/drive/MyDrive/KZN_Research_Colab/Stacked_4tile/Aligned_bands/S1_VV_pre_aligned.tif
  [OK] -> /content/drive/MyDrive/KZN_Research_Colab/Stacked_4tile/Aligned_bands/S1_VV_post_aligned.tif


In [ ]:
print('Resampling DEM ...\n')

dem_src = ROOT + 'DEM/Processed/DEM_4tile_30m_native.tif'
dem_dst = ALIGNED_DIR + 'DEM_aligned.tif'

if os.path.exists(dem_dst):
    print('  [SKIP - already done]')
else:
    resample_to_reference_file(dem_src, dem_dst, resampling=Resampling.bilinear)
    print(f'  [OK] -> {dem_dst}')

Resampling DEM ...

  [OK] -> /content/drive/MyDrive/KZN_Research_Colab/Stacked_4tile/Aligned_bands/DEM_aligned.tif


---
## Step 3: Verify All Aligned Bands Share the Same Grid

A quick check before combining anything — confirms every aligned file
has identical shape, CRS, and transform.

In [ ]:
aligned_files = sorted([f for f in os.listdir(ALIGNED_DIR) if f.endswith('.tif')])
print(f'{len(aligned_files)} aligned files found:\n')

all_consistent = True
for fname in aligned_files:
    with rasterio.open(ALIGNED_DIR + fname) as src:
        match = (src.shape == (ref_height, ref_width)) and (str(src.crs) == str(ref_crs))
        if not match:
            all_consistent = False
        print(f'  {fname}: shape={src.shape}  crs={src.crs}  [{"OK" if match else "MISMATCH"}]')

print()
print('All aligned bands consistent:', all_consistent)

9 aligned files found:

  DEM_aligned.tif: shape=(22139, 21576)  crs=EPSG:32736  [OK]
  S1_VV_post_aligned.tif: shape=(22139, 21576)  crs=EPSG:32736  [OK]
  S1_VV_pre_aligned.tif: shape=(22139, 21576)  crs=EPSG:32736  [OK]
  S2_B02_aligned.tif: shape=(22139, 21576)  crs=EPSG:32736  [OK]
  S2_B03_aligned.tif: shape=(22139, 21576)  crs=EPSG:32736  [OK]
  S2_B04_aligned.tif: shape=(22139, 21576)  crs=EPSG:32736  [OK]
  S2_B08_aligned.tif: shape=(22139, 21576)  crs=EPSG:32736  [OK]
  S2_B11_aligned.tif: shape=(22139, 21576)  crs=EPSG:32736  [OK]
  S2_B12_aligned.tif: shape=(22139, 21576)  crs=EPSG:32736  [OK]

All aligned bands consistent: True


---
## Step 4: Compute Spectral Indices (Windowed — Low Memory)

NDVI, NDWI, MNDWI are computed and written to disk by processing the
raster in horizontal strips (windows) rather than loading the full
~1.9GB array for each band at once. Each window is small enough to keep
total memory use modest regardless of the full image size.

In [ ]:
SCALE = 10000.0
eps = 1e-10
WINDOW_HEIGHT = 1000

def compute_index_windowed(band_a_path, band_b_path, formula, out_path, scale=SCALE):
    """
    formula: function(a, b) -> index array, where a and b are already
    scaled to true reflectance (divided by `scale`).
    Processes the raster in horizontal window strips to limit memory use.
    """
    with rasterio.open(band_a_path) as src_a, rasterio.open(band_b_path) as src_b:
        meta = src_a.meta.copy()
        meta.update({'dtype': 'float32', 'count': 1})
        with rasterio.open(out_path, 'w', **meta) as dst:
            for row_start in range(0, src_a.height, WINDOW_HEIGHT):
                rows = min(WINDOW_HEIGHT, src_a.height - row_start)
                window = Window(0, row_start, src_a.width, rows)
                a = src_a.read(1, window=window).astype(np.float32) / scale
                b = src_b.read(1, window=window).astype(np.float32) / scale
                idx = formula(a, b)
                dst.write(idx.astype(np.float32), 1, window=window)

print('Computing NDVI ...')
compute_index_windowed(
    ALIGNED_DIR + 'S2_B08_aligned.tif', ALIGNED_DIR + 'S2_B04_aligned.tif',
    lambda nir, red: (nir - red) / (nir + red + eps),
    ALIGNED_DIR + 'NDVI.tif'
)

print('Computing NDWI ...')
compute_index_windowed(
    ALIGNED_DIR + 'S2_B03_aligned.tif', ALIGNED_DIR + 'S2_B08_aligned.tif',
    lambda green, nir: (green - nir) / (green + nir + eps),
    ALIGNED_DIR + 'NDWI.tif'
)

print('Computing MNDWI ...')
compute_index_windowed(
    ALIGNED_DIR + 'S2_B03_aligned.tif', ALIGNED_DIR + 'S2_B11_aligned.tif',
    lambda green, swir1: (green - swir1) / (green + swir1 + eps),
    ALIGNED_DIR + 'MNDWI.tif'
)

print('\nIndices computed and saved.')

Computing NDVI ...
Computing NDWI ...
Computing MNDWI ...

Indices computed and saved.


---
## Step 5: Normalise DEM (Windowed)

DEM min/max are computed first via a low-memory pass (block statistics),
then applied window-by-window — avoids loading the full DEM array twice.

In [ ]:
dem_path = ALIGNED_DIR + 'DEM_aligned.tif'

# Pass 1: find min/max - but first CLIP negative values to 0.
# KwaZulu-Natal has no land below sea level; negative values found here
# (e.g. -25m at lat=-29.36, lon=31.30, located over open ocean) are
# nodata/void artifacts from the SRTM source over water, not real
# elevation. Clipping avoids letting these skew the normalisation range

dem_min, dem_max = np.inf, -np.inf
with rasterio.open(dem_path) as src:
    for row_start in range(0, src.height, WINDOW_HEIGHT):
        rows = min(WINDOW_HEIGHT, src.height - row_start)
        window = Window(0, row_start, src.width, rows)
        chunk = src.read(1, window=window)
        chunk = np.clip(chunk, 0, None)
        chunk_valid = chunk[np.isfinite(chunk)]
        if chunk_valid.size > 0:
            dem_min = min(dem_min, chunk_valid.min())
            dem_max = max(dem_max, chunk_valid.max())

print(f'DEM range after clipping negatives to 0: {dem_min:.1f}m to {dem_max:.1f}m')

# Pass 2: clip and normalise window-by-window, write to a new file
dem_norm_path = ALIGNED_DIR + 'DEM_normalised.tif'
with rasterio.open(dem_path) as src:
    meta = src.meta.copy()
    with rasterio.open(dem_norm_path, 'w', **meta) as dst:
        for row_start in range(0, src.height, WINDOW_HEIGHT):
            rows = min(WINDOW_HEIGHT, src.height - row_start)
            window = Window(0, row_start, src.width, rows)
            chunk = src.read(1, window=window).astype(np.float32)
            chunk = np.clip(chunk, 0, None)
            chunk_norm = (chunk - dem_min) / (dem_max - dem_min + eps)
            dst.write(chunk_norm, 1, window=window)

print('DEM clipped, normalised, and saved.')

DEM range after clipping negatives to 0: 0.0m to 2026.0m
DEM clipped, normalised, and saved.


---
## Step 6: Assemble the Final 12-Channel Stack (Windowed)

Reads all 12 single-band aligned files window-by-window and writes
directly into one multi-band output file — the full 12-channel array is
never held in memory at once, only one window's worth (12 x window_height
x width, which at WINDOW_HEIGHT=1000 is roughly 12 x 1000 x 21576 x 4
bytes ~= 1GB per window, comfortably within memory).

In [ ]:
OUT_DIR = ROOT + 'Stacked_4tile/'
WINDOW_HEIGHT = 1000

band_paths = [
    ALIGNED_DIR + 'S2_B02_aligned.tif',
    ALIGNED_DIR + 'S2_B03_aligned.tif',
    ALIGNED_DIR + 'S2_B04_aligned.tif',
    ALIGNED_DIR + 'S2_B08_aligned.tif',
    ALIGNED_DIR + 'S2_B11_aligned.tif',
    ALIGNED_DIR + 'S2_B12_aligned.tif',
    ALIGNED_DIR + 'NDVI.tif',
    ALIGNED_DIR + 'NDWI.tif',
    ALIGNED_DIR + 'MNDWI.tif',
    ALIGNED_DIR + 'S1_VV_pre_aligned.tif',
    ALIGNED_DIR + 'S1_VV_post_aligned.tif',
    ALIGNED_DIR + 'DEM_normalised.tif',
]
band_names = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12',
              'NDVI', 'NDWI', 'MNDWI', 'VV_pre', 'VV_post', 'DEM']

stack_path = OUT_DIR + 'flood_stack_4tile_v1.tif'

srcs = [rasterio.open(p) for p in band_paths]
meta = srcs[0].meta.copy()
meta.update({'count': 12, 'dtype': 'float32'})

with rasterio.open(stack_path, 'w', **meta) as dst:
    for i, name in enumerate(band_names, start=1):
        dst.set_band_description(i, name)

    for row_start in range(0, ref_height, WINDOW_HEIGHT):
        rows = min(WINDOW_HEIGHT, ref_height - row_start)
        window = Window(0, row_start, ref_width, rows)
        for band_idx, src in enumerate(srcs, start=1):
            chunk = src.read(1, window=window).astype(np.float32)
            chunk = np.nan_to_num(chunk, nan=0.0, posinf=0.0, neginf=0.0)
            dst.write(chunk, band_idx, window=window)
        if row_start % (WINDOW_HEIGHT * 5) == 0:
            print(f'  Processed rows {row_start}-{row_start+rows} of {ref_height}')

for s in srcs:
    s.close()

print(f'\nStack saved: {stack_path}')
print(f'Size: {os.path.getsize(stack_path) / 1e9:.2f} GB')

  Processed rows 0-1000 of 20983
  Processed rows 5000-6000 of 20983
  Processed rows 10000-11000 of 20983
  Processed rows 15000-16000 of 20983
  Processed rows 20000-20983 of 20983

Stack saved: /content/drive/MyDrive/KZN_Research_Colab/Stacked_4tile/flood_stack_4tile_v1.tif
Size: 22.93 GB


---
## Step 8: Summary